# Stage 5 — Type-I Rank-1 Codebook / Precoder Parity

Bu aşama **mevcut gerçek kullanım yolunu** taşır:

```matlab
generate_codebook(2,N1,N2,1,1)
selection_precoder(codebook,1,V)
```

Yani:

\[
XP=2,\qquad cb\_mode=1,\qquad nl=1.
\]

Test edilen array'ler:

\[
(1,1),(2,1),(4,1),(2,2),(4,2).
\]

Her biri için:

- tüm codebook numerik olarak karşılaştırılır,
- `i11/i12/i2` indeks haritası karşılaştırılır,
- MATLAB `V(:,1)` verilerek selector izole biçimde test edilir,
- aynı `H` üzerinde Python GPU SVD + selector end-to-end test edilir.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys, zipfile, shutil, json
import numpy as np
import pandas as pd
import torch

ROOT = Path('/content/drive/MyDrive/MyDrive/RIS')

MODULE = ROOT / 'ris_gpu_precoder_stage5.py'
if not MODULE.exists():
    MODULE = Path('/content/ris_gpu_precoder_stage5.py')

assert MODULE.exists(), (
    "ris_gpu_precoder_stage5.py dosyasını "
    "Drive RIS root'a veya /content altına yükle."
)

sys.path.insert(0,str(MODULE.parent))

from ris_gpu_precoder_stage5 import (
    compare_stage5_matlab_case,
    benchmark_selector,
)

print("CUDA:",torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :",torch.cuda.get_device_name(0))
print("Loaded:",MODULE)

## MATLAB golden suite

MATLAB'da:

```matlab
export_stage5_precoder_suite
```

çalıştır.

Sonra oluşan:

```text
stage5_precoder_golden.zip
```

dosyasını `/content` altına yükle.

In [ ]:
ZIP = Path('/content/stage5_precoder_golden.zip')
assert ZIP.exists(), "stage5_precoder_golden.zip dosyasını /content altına yükle."

EXTRACT = Path('/content/stage5_precoder_extract')
if EXTRACT.exists():
    shutil.rmtree(EXTRACT)
EXTRACT.mkdir(parents=True)

with zipfile.ZipFile(ZIP,'r') as zf:
    zf.extractall(EXTRACT)

mans = list(EXTRACT.rglob('manifest.csv'))
assert len(mans) == 1, mans

SUITE = mans[0].parent
manifest = pd.read_csv(mans[0])

display(manifest)

expected = {(1,1),(2,1),(4,1),(2,2),(4,2)}
found = set(zip(manifest['N1'].astype(int),manifest['N2'].astype(int)))
assert found == expected

print("PASS: expected array coverage")

In [ ]:
# DOUBLE / COMPLEX128 PARITY

device = 'cuda' if torch.cuda.is_available() else 'cpu'

rows = []

for _,r in manifest.iterrows():

    m = compare_stage5_matlab_case(
        str(SUITE/str(r['file'])),
        device=device,
        parity=True,
    )

    rows.append({
        'N1':int(r['N1']),
        'N2':int(r['N2']),
        'nT':int(r['nT']),
        **m
    })

df64 = pd.DataFrame(rows)
display(df64)

assert df64['codebook_index_exact'].all()
assert (df64['selector_matV_index_match_pct'] == 100.0).all()
assert (df64['svd_selector_index_match_pct'] == 100.0).all()

worst_numeric = max(
    df64['codebook_relFro'].max(),
    df64['selector_matV_W_relFro'].max(),
    df64['svd_V1_phaseInvariant_max'].max(),
)

print("Worst double numerical metric:",worst_numeric)

assert worst_numeric < 1e-10, (
    f"Stage 5 double parity failed: {worst_numeric:.3e}"
)

print("PASS: Stage 5 Type-I rank-1 double parity")

In [ ]:
# FLOAT32 / COMPLEX64 PRODUCTION SANITY

rows = []

for _,r in manifest.iterrows():

    m = compare_stage5_matlab_case(
        str(SUITE/str(r['file'])),
        device=device,
        parity=False,
    )

    rows.append({
        'N1':int(r['N1']),
        'N2':int(r['N2']),
        'nT':int(r['nT']),
        **m
    })

df32 = pd.DataFrame(rows)
display(df32)

# Codebook/selector from MATLAB V should remain exact in discrete indices.
assert df32['codebook_index_exact'].all()
assert (df32['selector_matV_index_match_pct'] == 100.0).all()

# SVD can be more precision-sensitive, but on these nondegenerate random
# matrices we expect the same selected codebook index.
assert (df32['svd_selector_index_match_pct'] == 100.0).all()

worst32 = max(
    df32['codebook_relFro'].max(),
    df32['selector_matV_W_relFro'].max(),
)

print("Worst float32 codebook/selector relative error:",worst32)

assert worst32 < 1e-5

print("PASS: Stage 5 float32 production sanity")

In [ ]:
# Optional selector throughput benchmark.
if torch.cuda.is_available():
    bench = benchmark_selector(
        N1=4,
        N2=2,
        batch_size=65536,
        repeats=10,
        device='cuda',
    )
    print(json.dumps(bench,indent=2))

## Stage 5 kabul kriteri

Stage 5 ancak aşağıdakilerin tamamı sağlanırsa kapanır:

\[
e_{\rm codebook}<10^{-10}
\]

\[
WIdx_{\rm MATLAB\ V}=WIdx_{\rm Python}=100\%
\]

\[
WIdx_{\rm MATLAB\ SVD}=WIdx_{\rm Python\ SVD}=100\%
\]

ve MATLAB/Python ilk right-singular vector'ları global fazdan bağımsız olarak aynı doğrultudadır.

Sonra geometry parity'ye geçeceğiz.